In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# we building an agent to to tell me what I can cook. 
# let him cook..... ?

# step one: upload the image

from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [4]:
print(uploader.value)

({'name': 'fridge.jpg', 'type': 'image/jpeg', 'size': 285141, 'content': <memory at 0x10a630ac0>, 'last_modified': datetime.datetime(2026, 2, 14, 17, 19, 41, 422000, tzinfo=datetime.timezone.utc)},)


In [5]:
# step two: encode the image in base64 for passing it to the agent.
# P.S. the image should not exceed a max of 5MB or it throws an error.
import base64

uploaded_file = uploader.value[0]

content_mv = uploaded_file["content"]

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [6]:
# step three: define tool to search the internet for recipes of the ingredients identified from the image
from tavily import TavilyClient
from langchain.tools import tool
from typing import Dict, Any

tavily_client = TavilyClient()

@tool
def search_web(query: str) -> Dict[str, Any]:
    """search the web for recipes using only the ingredients from the image."""
    return tavily_client.search(query)

In [7]:
# step four: create the agent
from langchain.agents import create_agent

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_web],
    system_prompt="You are a chef."
)

In [10]:
# step five: invoke the agent. 
agent.invoke(
    {"messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "tell me a few dishes I can make from the ingredients in my fridge"}, 
            {"type": "image", "base64": img_b64, "mime_type": "image/jpeg"}
        ]
    }]}
)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'tell me a few dishes I can make from the ingredients in my fridge'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}], additional_kwargs={}, response_metadata={}, id='7486016f-c460-4432-beca-6d4b2d8ace93'),
  AIMessage(content=[{'text': 'Looking at your fridge, I can see you have some great ingredients to work with! Let me search for some recipe ideas using what you have.', 'type': 'text'}, {'id': 'toolu_01CZA86XjXp3SMakz6hqMjXr', 'input': {'query': 'recipes with broccoli Brussels sprouts mushrooms grapes berries almond milk creamer eggs'}, 'name': 'search_web', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01FGuTttTy5VEcgrZaSUbgxe', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_